<a href="https://colab.research.google.com/github/eliza-aurora-carling/CFG-Assignments/blob/main/CFG_Assignment_4_Daunt_books.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [14]:
!pip install flask flask-ngrok pyngrok requests


In [15]:
# CFG Assignment 4
# Daunt Books Library API


!pip install -q flask requests

import sqlite3
import threading
import time
from datetime import datetime
from flask import Flask, request, jsonify
import requests
DB_FILE = "daunt_books.db"

def setup_database():
    """Create database, table, and populate with sample books."""
    conn = sqlite3.connect(DB_FILE)
    cursor = conn.cursor()

    cursor.execute("""
        CREATE TABLE IF NOT EXISTS books (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            title TEXT NOT NULL,
            author TEXT NOT NULL,
            genre TEXT,
            available INTEGER DEFAULT 1,
            borrower_name TEXT DEFAULT NULL,
            borrow_date TEXT DEFAULT NULL
        );
    """)

    cursor.execute("SELECT COUNT(*) FROM books;")
    count = cursor.fetchone()[0]

    if count == 0:
        sample_books = [
            ("A Room with a View", "E.M. Forster", "Fiction"),
            ("The Talented Mr Ripley", "Patricia Highsmith", "Thriller"),
            ("My Brilliant Friend", "Elena Ferrante", "Fiction"),
            ("The Silk Roads", "Peter Frankopan", "History"),
            ("Pachinko", "Min Jin Lee", "Fiction"),
            ("The Island of Missing Trees", "Elif Shafak", "Fiction"),
            ("In Patagonia", "Bruce Chatwin", "Travel"),
            ("The Song of Achilles", "Madeline Miller", "Fantasy"),
            ("Shuggie Bain", "Douglas Stuart", "Fiction"),
            ("Piranesi", "Susanna Clarke", "Fantasy")
        ]
        cursor.executemany(
            "INSERT INTO books (title, author, genre) VALUES (?, ?, ?);",
            sample_books
        )
        conn.commit()
        print(" Database created and populated with 10 sample books.")
    else:
        print(f"ℹ️  Database already exists with {count} books.")

    conn.close()

def execute_query(query, params=None, fetch_results=True):
    """
    Execute a SQL query with error handling.
    Returns list of dicts for SELECT, row count for UPDATE/INSERT.
    """
    conn = None
    cursor = None
    try:
        conn = sqlite3.connect(DB_FILE)
        conn.row_factory = sqlite3.Row
        cursor = conn.cursor()

        if params:
            cursor.execute(query, params)
        else:
            cursor.execute(query)

        if fetch_results:
            return [dict(row) for row in cursor.fetchall()]
        else:
            conn.commit()
            return cursor.rowcount

    except sqlite3.Error as e:
        print(f" Database error: {e}")
        return None
    finally:
        if cursor:
            cursor.close()
        if conn:
            conn.close()

# Flask api
app = Flask(__name__)

@app.route("/books", methods=["GET"])
def get_all_books():
    """ENDPOINT 1: Fetch all books with availability status."""
    try:
        query = "SELECT id, title, author, genre, available, borrower_name FROM books;"
        books = execute_query(query)

        if books is None:
            return jsonify({"error": "Database error occurred"}), 500

        if not books:
            return jsonify({"message": "No books found in the library"}), 200

        for book in books:
            book["available"] = bool(book["available"])

        return jsonify(books), 200

    except Exception as e:
        return jsonify({"error": str(e)}), 500


@app.route("/borrow", methods=["POST"])
def borrow_book():
    """ENDPOINT 2: Borrow a book from the library."""
    try:
        data = request.get_json(silent=True)
        if not data:
            return jsonify({"error": "No JSON data provided"}), 400

        book_id = data.get("book_id")
        borrower_name = data.get("borrower_name")

        if not book_id or not borrower_name:
            return jsonify({"error": "Missing 'book_id' or 'borrower_name'"}), 400

        # checking if the book is available  (SQL query type 1: 'SELECT' with WHERE)
        check_query = "SELECT id, title FROM books WHERE id = ? AND available = 1;"
        book = execute_query(check_query, (book_id,))

        if book is None:
            return jsonify({"error": "Database error"}), 500

        if not book:
            return jsonify({"error": f"Book ID {book_id} is not available or doesn't exist"}), 404

        # Mark the book as 'borrowed' (SQL query type 2: UPDATE)
        borrow_date = datetime.now().strftime("%d %B %Y at %H:%M")
        update_query = """
            UPDATE books
            SET available = 0, borrower_name = ?, borrow_date = ?
            WHERE id = ?;
        """
        affected = execute_query(update_query, (borrower_name, borrow_date, book_id), fetch_results=False)

        if not affected:
            return jsonify({"error": "Failed to borrow book"}), 500

        return jsonify({
            "message": f"'{book[0]['title']}' has been borrowed by {borrower_name}.",
            "book_id": book_id,
            "borrow_date": borrow_date,
            "reminder": "Please return within 3 weeks. Late returns incur a £0.50/day fee."
        }), 200

    except Exception as e:
        return jsonify({"error": str(e)}), 500


@app.route("/available", methods=["GET"])
def get_available_books():
    """ENDPOINT 3: Show available books on the shelves, with optional genre filter."""
    try:
        genre = request.args.get("genre")

        if genre:
            query = """
                SELECT id, title, author, genre
                FROM books
                WHERE available = 1 AND LOWER(genre) = LOWER(?)
                ORDER BY title;
            """
            books = execute_query(query, (genre,))
        else:
            # All available
            query = """
                SELECT id, title, author, genre
                FROM books
                WHERE available = 1
                ORDER BY title;
            """
            books = execute_query(query)

        if books is None:
            return jsonify({"error": "Database error"}), 500

        if not books:
            msg = "No available books found"
            if genre:
                msg += f" in {genre}"
        else:
            msg = f"Found {len(books)} available book(s)"
            if genre:
                msg += f" in {genre}"

        return jsonify({
            "available_count": len(books) if books else 0,
            "message": msg,
            "books": books if books else []
        }), 200

    except Exception as e:
        return jsonify({"error": str(e)}), 500


@app.route("/", methods=["GET"])
def home():
    """Welcome endpoint."""
    return jsonify({
        "message": "Welcome to the Daunt Books Library API",
        "location": "84 Marylebone High Street, London W1U 4QW",
        "endpoints": {
            "GET /books": "View all books",
            "POST /borrow": "Borrow a book (requires book_id and borrower_name)",
            "GET /available": "View available books (optional ?genre= parameter)"
        }
    })

# database
setup_database()


LOCAL_URL = "http://127.0.0.1:5000"

def start_flask():
    app.run(host="127.0.0.1", port=5000, use_reloader=False)

thread = threading.Thread(target=start_flask, daemon=True)
thread.start()
time.sleep(3)
print("Flask server is running in the background on port 5000\n")

def display_welcome():
    """Display the Daunt Books welcome banner."""
    print("       WELCOME TO DAUNT BOOKS :) ")
    print("       84 Marylebone High Street")
    print("              London W1U 4QW")
    print(f"       Today: {datetime.now().strftime('%d %B %Y')}")

def browse_all_books():
    """
    Simulate customer browsing all books.
    Uses: GET /books
    """
    print(" " * 50)
    print("BROWSING ALL BOOKS IN OUR SHOP (Daunt Books)")
    print(" " * 50)
    try:
        r = requests.get(f"{LOCAL_URL}/books", timeout=10)
        if r.status_code == 200:
            books = r.json()
            if isinstance(books, list):
                for b in books:
                    if b["available"]:
                        status = "AVAILABLE"
                    else:
                        status = f"ON LOAN (to {b.get('borrower_name', 'Unknown')})"
                    print(f"\n  [{b['id']}] {b['title']}")
                    print(f"       by {b['author']}")
                    print(f"       Genre: {b['genre']} | {status}")
            else:
                print(f"  ℹ️  {books.get('message', 'No books found')}")
        else:
            print(f"   Error: {r.json().get('error', 'Unknown error')}")
    except requests.exceptions.ConnectionError:
        print("   Can't connect to the API. Is the server running?")
    except Exception as e:
        print(f"   unexpected error: {e}")


def borrow_a_book(book_id, borrower_name):
    """
    Simulate customer borrowing a book at the counter.
    Uses: POST /borrow
    """
    print(" " * 50)
    print(f" BORROWING: Book ID {book_id}")
    print(" " * 50)
    print(f"  Customer: {borrower_name}")
    try:
        payload = {"book_id": book_id, "borrower_name": borrower_name}
        r = requests.post(f"{LOCAL_URL}/borrow", json=payload, timeout=10)

        if r.status_code == 200:
            d = r.json()
            print(f"   {d['message']}")
            print(f"   Borrowed on: {d['borrow_date']}")
            print(f"    {d['reminder']}")
        elif r.status_code == 404:
            print(f"   {r.json()['error']}")
        else:
            print(f"   {r.json().get('error', 'Failed to borrow book')}")
    except requests.exceptions.ConnectionError:
        print("  Cannot connect to API.")
    except Exception as e:
        print(f"  Unexpected error: {e}")


def check_available_books(genre=None):
    """
    Simulate customer checking what's on the shelves.
    Uses: GET /available?genre=...
    """
    print(" " * 50)
    if genre:
        print(f" Checking shelves: {genre.upper()}")
    else:
        print("📋Checking all of the available books")
    print(" " * 50)
    try:
        url = f"{LOCAL_URL}/available"
        if genre:
            url += f"?genre={genre}"
        r = requests.get(url, timeout=10)

        if r.status_code == 200:
            d = r.json()
            print(f"   {d['message']}")
            if d.get("books"):
                for b in d["books"]:
                    print(f"     [{b['id']}] {b['title']} by {b['author']} ({b['genre']})")
            else:
                print("  The shelf is empty.")
        else:
            print(f"   {r.json().get('error', 'Error')}")
    except requests.exceptions.ConnectionError:
        print("  Can't connect to API.")
    except Exception as e:
        print(f"  Unexpected error: {e}")


def run():
    """
    Main simulation function.
    Walks through a complete customer journey at Daunt Books.
    """
    display_welcome()

    # scene 1
    print("\n A customer walks in and browses the collection...")
    browse_all_books()

    # scene 2
    print("\n Customer asks: 'What fiction books are on the shelves?'")
    check_available_books(genre="Fiction")

    # scene 3
    print("\n customer picks 'My Brilliant Friend' and heads to the counter...")
    borrow_a_book(book_id=3, borrower_name="Zadie Smith")

    # scene 4
    print("\n Another customer checks the fiction bookshelves...")
    check_available_books(genre="Fiction")

    # scene 5
    print("\n Later: staff member reviews the full collection...")
    browse_all_books()

    # final goodbye message
    print("   Thank you for visiting DAUNT BOOKS")
    print(f"\n    Visit us: 84 Marylebone High Street, London W1U 4QW")
    print(f"   Opening hours: Mon-Sat 9:00-19:30 | Sun 11:00-18:00")
    print(f"   www.dauntbooks.co.uk\n")



time.sleep(1)
run()
print("   QUICK MANUAL ENDPOINT TESTS")


print("\n>>> TEST 1: Welcome Page")
r = requests.get(f"{LOCAL_URL}/", timeout=5)
if r.status_code == 200:
    print(f"   Status: {r.status_code} done")
    print(f"   Message: {r.json()['message']}")

print("\n>>> TEST 2: GET /books (showing first 2)")
r = requests.get(f"{LOCAL_URL}/books", timeout=5)
if r.status_code == 200:
    books = r.json()
    for b in books[:2]:
        status = "Available" if b["available"] else "On Loan"
        print(f"   [{b['id']}] {b['title']} - {status}")

print("\n>>> TEST 3: GET /available?genre=Travel")
r = requests.get(f"{LOCAL_URL}/available?genre=Travel", timeout=5)
if r.status_code == 200:
    print(f"   Status: {r.status_code} ")
    print(f"   {r.json()['message']}")
    for b in r.json().get("books", []):
        print(f"   [{b['id']}] {b['title']}")

print("\n>>> TEST 4: POST /borrow (borrowing book 5)")
r = requests.post(f"{LOCAL_URL}/borrow",
                 json={"book_id": 5, "borrower_name": "Test Customer"},
                 timeout=5)
if r.status_code == 200:
    print(f"   Status: {r.status_code} ")
    print(f"   {r.json()['message']}")
elif r.status_code == 404:
    print(f"   Status: {r.status_code} - Book already borrowed")

print("\n>>> TEST 5: POST /borrow (trying to borrow same book again)")
r = requests.post(f"{LOCAL_URL}/borrow",
                 json={"book_id": 5, "borrower_name": "Another Person"},
                 timeout=5)
print(f"   Status: {r.status_code}")
print(f"   Response: {r.json()['error']}")


print(" ALL TESTS COMPLETE!")
print("\n Summary of what this demonstrates:")
print("   • GET /books - Retrieves all books (ENDPOINT 1)")
print("   • POST /borrow - Borrows a book (ENDPOINT 2)")
print("   • GET /available?genre=X - Filters available books (ENDPOINT 3)")
print("   • Client simulation with run() function")
print("   • SQLite database with 1 table (books)")
print("   • Two different SQL query types (SELECT + UPDATE)")
print("   • Error handling throughout")
print("   • Welcome messages and formatted output")
print("\n The API is running successfully in Colab!")

ℹ️  Database already exists with 10 books.
 * Serving Flask app '__main__'
 * Debug mode: off


Address already in use
Port 5000 is in use by another program. Either identify and stop that program, or start the server with a different port.


Flask server is running in the background on port 5000



INFO:werkzeug:127.0.0.1 - - [11/May/2026 13:53:37] "GET /books HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [11/May/2026 13:53:37] "GET /available?genre=Fiction HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [11/May/2026 13:53:37] "POST /borrow HTTP/1.1" 404 -
INFO:werkzeug:127.0.0.1 - - [11/May/2026 13:53:37] "GET /available?genre=Fiction HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [11/May/2026 13:53:37] "GET /books HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [11/May/2026 13:53:37] "GET / HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [11/May/2026 13:53:37] "GET /books HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [11/May/2026 13:53:37] "GET /available?genre=Travel HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [11/May/2026 13:53:37] "POST /borrow HTTP/1.1" 404 -
INFO:werkzeug:127.0.0.1 - - [11/May/2026 13:53:37] "POST /borrow HTTP/1.1" 404 -


       WELCOME TO DAUNT BOOKS :) 
       84 Marylebone High Street
              London W1U 4QW
       Today: 11 May 2026

 A customer walks in and browses the collection...
                                                  
BROWSING ALL BOOKS IN OUR SHOP (Daunt Books)
                                                  

  [1] A Room with a View
       by E.M. Forster
       Genre: Fiction | AVAILABLE

  [2] The Talented Mr Ripley
       by Patricia Highsmith
       Genre: Thriller | AVAILABLE

  [3] My Brilliant Friend
       by Elena Ferrante
       Genre: Fiction | ON LOAN (to Zadie Smith)

  [4] The Silk Roads
       by Peter Frankopan
       Genre: History | AVAILABLE

  [5] Pachinko
       by Min Jin Lee
       Genre: Fiction | ON LOAN (to Test Customer)

  [6] The Island of Missing Trees
       by Elif Shafak
       Genre: Fiction | AVAILABLE

  [7] In Patagonia
       by Bruce Chatwin
       Genre: Travel | AVAILABLE

  [8] The Song of Achilles
       by Madeline Miller
       G